In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv("/kaggle/input/datasets/abbassu/house-price-prediction-dataset/house_pricing_dataset.csv")

# drop unnecessary structural and environmental columns
cols_to_drop = [
    "Id", "house_id", "school_rating", "crime_rate", 
    "distance_to_cbd_km", "renovation_score", "energy_rating"
]

# Drop only if they exist in the dataframe to avoid errors
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

print(f"Dataset Shape after dropping extra columns: {df.shape}")
df.head()

In [ ]:

target_col = "sale_price_usd" if "sale_price_usd" in df.columns else "Price"

X = df.drop(columns=[target_col])
y = df[target_col]

# 80% Train, 20% Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

In [ ]:

num_cols = X_train_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_clean.select_dtypes(include=['object', 'category']).columns.tolist()

#
for col in num_cols:
    median_val = X_train_clean[col].median()
    X_train_clean[col] = X_train_clean[col].fillna(median_val)
    X_test_clean[col] = X_test_clean[col].fillna(median_val)


for col in cat_cols:
    if len(X_train_clean[col].mode()) > 0:
        mode_val = X_train_clean[col].mode()[0]
        X_train_clean[col] = X_train_clean[col].fillna(mode_val)
        X_test_clean[col] = X_test_clean[col].fillna(mode_val)

print("✅ Imputation Complete! No missing values left.")

In [ ]:
if len(cat_cols) > 0:
   
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    
    encoded_train = encoder.fit_transform(X_train_clean[cat_cols])
    encoded_test = encoder.transform(X_test_clean[cat_cols])
    
    encoded_cols = encoder.get_feature_names_out(cat_cols)
    
    df_encoded_train = pd.DataFrame(encoded_train, columns=encoded_cols, index=X_train_clean.index)
    df_encoded_test = pd.DataFrame(encoded_test, columns=encoded_cols, index=X_test_clean.index)
    
    # 💡 MULTICOLLINEARITY FIX (Dummy Variable Trap):
   
    cols_to_drop = []
    for cat in cat_cols:
        
        matching_cols = [col for col in encoded_cols if col.startswith(f"{cat}_")]
        if matching_cols:
            cols_to_drop.append(matching_cols[0]) 
            
    df_encoded_train = df_encoded_train.drop(columns=cols_to_drop)
    df_encoded_test = df_encoded_test.drop(columns=cols_to_drop)
    

    X_train_clean = X_train_clean.drop(columns=cat_cols).join(df_encoded_train)
    X_test_clean = X_test_clean.drop(columns=cat_cols).join(df_encoded_test)

print(f"✅ One-Hot Encoding Done safely! Total features now: {X_train_clean.shape[1]}")

In [ ]:
scaler = StandardScaler()


X_train_clean[num_cols] = scaler.fit_transform(X_train_clean[num_cols])
X_test_clean[num_cols] = scaler.transform(X_test_clean[num_cols])

print("✅ Feature Scaling Applied to Continuous Variables.")

In [ ]:
# मॉडल को इनिशियलाइज और फिट करें
lr_model = LinearRegression()
lr_model.fit(X_train_clean, y_train)

# प्रेडिक्शन
y_pred = lr_model.predict(X_test_clean)

# मेट्रिक्स कैलकुलेट करें
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=========================================")
print("      FINAL MODEL PERFORMANCE METRICS    ")
print("=========================================")
print(f"Mean Absolute Error (MAE):     $ {mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): $ {rmse:,.2f}")
print(f"R-squared Score (R²):           {r2:.4f}")
print("=========================================\n")

# फीचर कोइफिशिएंट्स (Coefficients) देखें
coefficients = pd.DataFrame({
    "Feature": X_train_clean.columns,
    "Coefficient": lr_model.coef_
}).sort_values(by="Coefficient", ascending=False)

print("--- Feature Importance (Coefficients) ---")
print(coefficients.to_string(index=False))

In [7]:
import pickle

# Save the trained model
with open('lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

# Save your scaler (if you used one for numerical columns)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Artifacts saved! Download 'lr_model.pkl' and 'scaler.pkl' from your Kaggle output directory.")

Artifacts saved! Download 'lr_model.pkl' and 'scaler.pkl' from your Kaggle output directory.
